# risk-management_001

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (106).ipynb`

| Field | Value |
|---|---|
| Section | `risk_management` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 12 |
| Detected functions | weighted_moving_average, calculate_tmf, calculate_vwap, calculate_adx, process_ticker, apply_indicators_to_tickers, backtest_strategy_with_vwap_adx, calculate_adx, calculate_vwap, run_strategy_with_params, plot_kalman_signals, weighted_moving_average |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/filtered_data.csv')
data

In [ ]:
data = data[data['Ticker'] == 'RELIANCE']
data

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# Configuration variables in a dictionary
config = {
    'BROKERAGE': 0.001,         # Brokerage fee
    'TRADE_CAPITAL': 100000,    # Capital for each trade
    'TAKE_PROFIT': 0.015,       # Take profit percentage
    'STOP_LOSS': 0.02,          # Stop loss percentage
    'ADX_LENGTH': 14,           # ADX calculation length (default 14)
    'TMF_LENGTH': 21,           # Twiggs Money Flow length
    'ADX_THRESHOLD': 25,        # ADX threshold for trend confirmation
    'VWAP_ENABLED': True,       # If VWAP should be used for entry condition
    'TMF_ENABLED': True,        # If TMF should be used for entry condition
    'START_TIME': "12:00",      # Start time for trade execution (can be adjusted)
    'END_TIME': "15:28",        # End time for trade execution (can be adjusted)
    'EXCLUDE_DAY': 3,           # Day to exclude from trading (0: Monday, 1: Tuesday, etc.)
    'ADX_ENABLED': True         # If ADX should be used for trend confirmation
}

# Vectorized Weighted Moving Average (WiMA)
def weighted_moving_average(series, length):
    weights = np.arange(1, length + 1)
    wma = np.convolve(series, weights[::-1], mode='valid') / weights.sum()
    return np.concatenate([np.full(length - 1, np.nan), wma])

# Calculate Twiggs Money Flow (TMF)
def calculate_tmf(data, length):
    high, low, close, volume = data['high'].values, data['low'].values, data['close'].values, data['volume'].values
    prev_close = np.roll(close, 1)
    tr_h = np.maximum(prev_close, high)
    tr_l = np.minimum(prev_close, low)
    tr_c = tr_h - tr_l

    # Adjusted Volume
    adv = volume * ((close - tr_l) - (tr_h - close)) / np.where(tr_c == 0, np.nan, tr_c)
    adv[0] = 0  # First row adjustment

    # Weighted moving averages
    wmV = weighted_moving_average(volume, length)
    wmA = weighted_moving_average(adv, length)

    # Calculate TMF
    tmf = np.where(wmV == 0, 0, wmA / wmV)
    data['TMF'] = tmf
    return data

# Calculate VWAP (Volume Weighted Average Price)
def calculate_vwap(data):
    price = (data['high'] + data['low'] + data['close']) / 3
    vwap = (price * data['volume']).cumsum() / data['volume'].cumsum()
    data['VWAP'] = vwap
    return data

# Calculate ADX, +DI, and -DI
def calculate_adx(data, length=14):
    high = data['high']
    low = data['low']
    close = data['close']

    # True Range
    tr = pd.DataFrame({
        'tr1': high - low,
        'tr2': abs(high - close.shift(1)),
        'tr3': abs(low - close.shift(1))
    })
    tr = tr.max(axis=1)

    # +DM and -DM
    pdm = high.diff().clip(lower=0)
    ndm = -low.diff().clip(lower=0)

    # Smoothed True Range, +DM, and -DM
    tr_smooth = tr.rolling(window=length).sum()
    pdm_smooth = pdm.rolling(window=length).sum()
    ndm_smooth = ndm.rolling(window=length).sum()

    # +DI and -DI
    plus_di = 100 * pdm_smooth / tr_smooth
    minus_di = 100 * ndm_smooth / tr_smooth

    # ADX
    dx = 100 * (abs(plus_di - minus_di) / (plus_di + minus_di))
    adx = dx.rolling(window=length).mean()

    data['ADX'] = adx
    data['+DI'] = plus_di
    data['-DI'] = minus_di

    return data

# Process a single Ticker
def process_ticker(data, config):
    if config['TMF_ENABLED']:
        data = calculate_tmf(data, config['TMF_LENGTH'])
    if config['VWAP_ENABLED']:
        data = calculate_vwap(data)
    if config['ADX_ENABLED']:
        data = calculate_adx(data, config['ADX_LENGTH'])
    return data

# Apply indicators to all tickers with progress bar
def apply_indicators_to_tickers(data, config):
    tickers = data['Ticker'].unique()
    processed_data = [process_ticker(data[data['Ticker'] == ticker].copy(), config) for ticker in tqdm(tickers, desc="Processing Tickers")]
    return pd.concat(processed_data, ignore_index=True)

# Backtest strategy with TMF, VWAP, and ADX
def backtest_strategy_with_vwap_adx(data, config):
    tradebook = []

    data['date'] = pd.to_datetime(data['date'])
    data = data.sort_values(by=['Ticker', 'date'])

    # Prepare columns for take profit and stop loss
    data['tp_price_long'] = data['close'] * (1 + config['TAKE_PROFIT'])
    data['sl_price_long'] = data['close'] * (1 - config['STOP_LOSS'])
    data['tp_price_short'] = data['close'] * (1 - config['TAKE_PROFIT'])
    data['sl_price_short'] = data['close'] * (1 + config['STOP_LOSS'])

    for ticker, ticker_data in tqdm(data.groupby('Ticker'), desc="Backtesting Tickers"):
        ticker_data = ticker_data.reset_index(drop=True)
        position, entry_price, entry_time, quantity, cum_pnl = None, 0, None, 0, 0

        for i, row in ticker_data.iterrows():
            current_time = row['date'].time()

            # Skip excluded day (default: Tuesday)
            if row['date'].weekday() == config['EXCLUDE_DAY']:
                continue

            # Entry condition
            if position is None and current_time <= pd.to_datetime(config['START_TIME']).time():
                if (row['TMF'] > 0.25 and row['close'] > row['VWAP'] and row['ADX'] > config['ADX_THRESHOLD']) if config['TMF_ENABLED'] else False:  # Long condition with ADX trend confirmation
                    position = 'long'
                    entry_price = row['close']
                    entry_time = row['date']
                    quantity = config['TRADE_CAPITAL'] / entry_price
                elif (row['TMF'] < -0.25 and row['close'] < row['VWAP'] and row['ADX'] > config['ADX_THRESHOLD']) if config['TMF_ENABLED'] else False:  # Short condition with ADX trend confirmation
                    position = 'short'
                    entry_price = row['close']
                    entry_time = row['date']
                    quantity = config['TRADE_CAPITAL'] / entry_price

            # Exit condition
            if position is not None:
                exit_reason, exit_price = None, 0

                if position == 'long' and row['high'] >= row['tp_price_long']:
                    exit_price = row['tp_price_long']
                    exit_reason = 'Take Profit'
                elif position == 'long' and row['low'] <= row['sl_price_long']:
                    exit_price = row['sl_price_long']
                    exit_reason = 'Stop Loss'
                elif position == 'short' and row['low'] <= row['tp_price_short']:
                    exit_price = row['tp_price_short']
                    exit_reason = 'Take Profit'
                elif position == 'short' and row['high'] >= row['sl_price_short']:
                    exit_price = row['sl_price_short']
                    exit_reason = 'Stop Loss'

                # End of day exit
                if current_time >= pd.to_datetime(config['END_TIME']).time() and exit_reason is None:
                    exit_price = row['close']
                    exit_reason = 'End of Day'

                if exit_reason:
                    trade_cost = entry_price * quantity * config['BROKERAGE']
                    exit_cost = exit_price * quantity * config['BROKERAGE']
                    pnl = (exit_price - entry_price) * quantity if position == 'long' else \
                          (entry_price - exit_price) * quantity
                    pnl -= (trade_cost + exit_cost)
                    cum_pnl += pnl

                    tradebook.append({
                        'Ticker': ticker,
                        'Entry Time': entry_time,
                        'Exit Time': row['date'],
                        'Entry Price': entry_price,
                        'Exit Price': exit_price,
                        'PnL': pnl,
                        'Exit Reason': exit_reason,
                        'Cumulative PnL': cum_pnl
                    })

                    position, entry_price, entry_time, quantity = None, 0, None, 0

    # Save to CSV
    tradebook_df = pd.DataFrame(tradebook)
    tradebook_df.to_csv('/content/drive/MyDrive/tradebook.csv', index=False)
    print(tradebook_df)

# Main Execution
data_with_indicators = apply_indicators_to_tickers(data, config)
backtest_strategy_with_vwap_adx(data_with_indicators, config)


In [ ]:
tradebook_df

In [ ]:
import itertools
import pandas as pd
import numpy as np
from tqdm import tqdm

# Function to calculate ADX, +DI, and -DI
def calculate_adx(data, length=14):
    high = data['high']
    low = data['low']
    close = data['close']

    # True Range
    tr = pd.DataFrame({
        'tr1': high - low,
        'tr2': abs(high - close.shift(1)),
        'tr3': abs(low - close.shift(1))
    })
    tr = tr.max(axis=1)

    # +DM and -DM
    pdm = high.diff().clip(lower=0)
    ndm = -low.diff().clip(lower=0)

    # Smoothed True Range, +DM, and -DM
    tr_smooth = tr.rolling(window=length).sum()
    pdm_smooth = pdm.rolling(window=length).sum()
    ndm_smooth = ndm.rolling(window=length).sum()

    # +DI and -DI
    plus_di = 100 * pdm_smooth / tr_smooth
    minus_di = 100 * ndm_smooth / tr_smooth

    # ADX
    dx = 100 * (abs(plus_di - minus_di) / (plus_di + minus_di))
    adx = dx.rolling(window=length).mean()

    data['ADX'] = adx
    data['+DI'] = plus_di
    data['-DI'] = minus_di

    return data

# Function to calculate VWAP
def calculate_vwap(data):
    price = (data['high'] + data['low'] + data['close']) / 3
    vwap = (price * data['volume']).cumsum() / data['volume'].cumsum()
    data['VWAP'] = vwap
    return data

# Function to run the strategy with different parameters
def run_strategy_with_params(data, tmf_length, tp_level, sl_level, adx_threshold):
    """
    Run the backtest strategy with specified parameters and return performance metrics.

    Args:
        data (DataFrame): Market data.
        tmf_length (int): Length for TMF calculation.
        tp_level (float): Take-profit level (as a fraction, e.g., 0.015).
        sl_level (float): Stop-loss level (as a fraction, e.g., 0.015).
        adx_threshold (float): ADX threshold for trend strength (e.g., 25).

    Returns:
        dict: Performance metrics.
    """
    # Apply TMF calculation
    data_with_tmf = calculate_tmf(data.copy(), tmf_length)

    # Apply VWAP calculation
    data_with_vwap = calculate_vwap(data_with_tmf)

    # Apply ADX calculation
    data_with_adx = calculate_adx(data_with_vwap)

    tradebook = []
    BROKERAGE = 0.001
    TRADE_CAPITAL = 100000

    # Initialize cumulative PnL
    cum_pnl = 0
    data_with_adx['date'] = pd.to_datetime(data_with_adx['date'])

    # Precompute take-profit and stop-loss levels
    data_with_adx['tp_price_long'] = data_with_adx['close'] * (1 + tp_level)
    data_with_adx['sl_price_long'] = data_with_adx['close'] * (1 - sl_level)
    data_with_adx['tp_price_short'] = data_with_adx['close'] * (1 - tp_level)
    data_with_adx['sl_price_short'] = data_with_adx['close'] * (1 + sl_level)

    for ticker in data_with_adx['Ticker'].unique():
        ticker_data = data_with_adx[data_with_adx['Ticker'] == ticker].copy().sort_values(by='date')
        position, entry_price, entry_time, quantity = None, 0, None, 0

        for i, row in ticker_data.iterrows():
            current_time = row['date'].time()
            if row['date'].weekday() == 1:  # Skip Thursdays
                continue

            # Entry signals: Use ADX to filter trades
            if position is None and current_time <= pd.Timestamp("11:00").time():
                if row['TMF'] > 0.2499 and row['close'] > row['VWAP'] and row['ADX'] > adx_threshold:  # Long condition
                    position = 'long'
                    entry_price = row['close']
                    entry_time = row['date']
                    quantity = TRADE_CAPITAL / entry_price
                elif row['TMF'] < -0.2499 and row['close'] < row['VWAP'] and row['ADX'] > adx_threshold:  # Short condition
                    position = 'short'
                    entry_price = row['close']
                    entry_time = row['date']
                    quantity = TRADE_CAPITAL / entry_price

            # Exit signals
            if position is not None:
                exit_reason, exit_price = None, 0
                if position == 'long':
                    if row['high'] >= row['tp_price_long']:
                        exit_price = row['tp_price_long']
                        exit_reason = 'Take Profit'
                    elif row['low'] <= row['sl_price_long']:
                        exit_price = row['sl_price_long']
                        exit_reason = 'Stop Loss'
                elif position == 'short':
                    if row['low'] <= row['tp_price_short']:
                        exit_price = row['tp_price_short']
                        exit_reason = 'Take Profit'
                    elif row['high'] >= row['sl_price_short']:
                        exit_price = row['sl_price_short']
                        exit_reason = 'Stop Loss'

                if current_time.hour == 15 and current_time.minute == 28 and exit_reason is None:
                    exit_price = row['close']
                    exit_reason = 'End of Day'

                if exit_reason:
                    exit_time = row['date']
                    trade_cost = entry_price * quantity * BROKERAGE
                    exit_cost = exit_price * quantity * BROKERAGE
                    pnl = ((exit_price - entry_price) * quantity if position == 'long' else
                           (entry_price - exit_price) * quantity) - (trade_cost + exit_cost)

                    cum_pnl += pnl
                    tradebook.append({
                        'Ticker': ticker,
                        'PnL': pnl,
                        'Exit Reason': exit_reason
                    })

                    position, entry_price, entry_time, quantity = None, 0, None, 0

    # Calculate performance metrics
    total_trades = len(tradebook)
    winning_trades = sum(1 for t in tradebook if t['PnL'] > 0)
    losing_trades = sum(1 for t in tradebook if t['PnL'] <= 0)
    total_pnl = sum(t['PnL'] for t in tradebook)
    win_rate = (winning_trades / total_trades) if total_trades > 0 else 0

    return {
        'TMF Length': tmf_length,
        'Take Profit': tp_level,
        'Stop Loss': sl_level,
        'ADX Threshold': adx_threshold,
        'Total Trades': total_trades,
        'Winning Trades': winning_trades,
        'Losing Trades': losing_trades,
        'Total PnL': total_pnl,
        'Win Rate': win_rate
    }

# Define parameter grid for robustness testing
tmf_lengths = [67]
tp_levels = [0.015]
sl_levels = [0.02]
adx_thresholds = [25, 30]  # New ADX threshold values to test

# Generate all parameter combinations
parameter_combinations = list(itertools.product(tmf_lengths, tp_levels, sl_levels, adx_thresholds))

# Run robustness testing with progress bar
results = []
for params in tqdm(parameter_combinations, desc="Running Strategy Backtests"):
    tmf_length, tp_level, sl_level, adx_threshold = params
    result = run_strategy_with_params(data, tmf_length, tp_level, sl_level, adx_threshold)
    results.append(result)

# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Save results to CSV
results_df.to_csv('/content/drive/MyDrive/robustness_results_vwap_adx.csv', index=False)

# Print top-performing parameter sets
print(results_df.sort_values(by='Total PnL', ascending=False).head(10))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import plotly.graph_objects as go

def plot_kalman_signals(data, tradebook):
    """
    Plot the Kalman filter signals, close price, and trades using Plotly.

    Args:
        data (DataFrame): Data containing the 'close', 'Kalman', and signals.
        tradebook (DataFrame): Tradebook containing the entry and exit details.
    """
    # Create figure
    fig = go.Figure()





    # Plot close prices
    fig.add_trace(go.Scatter(
        x=data['date'],
        y=data['close'],
        mode='lines',
        name='Close Price',
        line=dict(color='blue', width=2)
    ))

    # Plot Kalman filter values
    fig.add_trace(go.Scatter(
        x=data['date'],
        y=data['Kalman'],
        mode='lines',
        name='Kalman Filter',
        line=dict(color='orange', dash='dot')
    ))

    # Plot buy and sell signals
    buy_signals = tradebook[tradebook['PnL'] > 0]
    sell_signals = tradebook[tradebook['PnL'] < 0]

    fig.add_trace(go.Scatter(
        x=buy_signals['Entry Time'],
        y=buy_signals['Entry Price'],
        mode='markers',
        name='Buy Signals',
        marker=dict(symbol='triangle-up', size=10, color='green'),
    ))

    fig.add_trace(go.Scatter(
        x=sell_signals['Entry Time'],
        y=sell_signals['Entry Price'],
        mode='markers',
        name='Sell Signals',
        marker=dict(symbol='triangle-down', size=10, color='red'),
    ))

    # Add lines connecting entry and exit
    for _, trade in tradebook.iterrows():
        color = 'green' if trade['PnL'] > 0 else 'red'
        fig.add_trace(go.Scatter(
            x=[trade['Entry Time'], trade['Exit Time']],
            y=[trade['Entry Price'], trade['Exit Price']],
            mode='lines',
            line=dict(color=color, width=2),
            showlegend=False
        ))

    # Layout configuration
    fig.update_layout(
        title='Kalman Filter: Buy and Sell Signals',
        xaxis_title='Date',
        yaxis_title='Price',
        legend_title='Legend',
        template='plotly_dark',
        height=600,
        width=1400
    )

    # Show plot
    fig.show()

# Assuming 'tradebook_df' is the tradebook DataFrame and 'data_with_indicators' is the data with Kalman filter
tradebook_df = pd.read_csv('/content/drive/MyDrive/tradebook.csv')  # Load tradebook
plot_kalman_signals(data_with_indicators, tradebook_df)


In [ ]:
import pandas as pd
import numpy as np
import itertools
from tqdm import tqdm  # For the progress bar

# Function to calculate Weighted Moving Average (WMA)
def weighted_moving_average(series, length):
    weights = np.arange(1, length + 1)
    wma = np.convolve(series, weights[::-1], mode='valid') / weights.sum()
    return np.concatenate([np.full(length - 1, np.nan), wma])

# Function to calculate Twiggs Money Flow (TMF)
def calculate_tmf(data, length=21):
    close, high, low, volume = data['close'], data['high'], data['low'], data['volume']

    tr_h = np.maximum(close.shift(1), high)
    tr_l = np.minimum(close.shift(1), low)
    tr_c = tr_h - tr_l

    adv = volume * ((close - tr_l) - (tr_h - close)) / np.where(tr_c == 0, np.nan, tr_c)
    adv.fillna(0, inplace=True)

    wmV = weighted_moving_average(volume.values, length)
    wmA = weighted_moving_average(adv.values, length)

    tmf = np.where(wmV == 0, 0, wmA / wmV)
    data['TMF'] = tmf

    return data

# Manual Kalman Filter Implementation
def calculate_kalman_filter_manual(series, process_noise, measurement_noise, initial_estimate=None):
    n = len(series)
    estimates = np.zeros(n)
    variances = np.zeros(n)

    estimates[0] = initial_estimate if initial_estimate is not None else series.iloc[0]
    variances[0] = 1

    for t in range(1, n):
        prediction = estimates[t - 1]
        variance_prediction = variances[t - 1] + process_noise

        kalman_gain = variance_prediction / (variance_prediction + measurement_noise)
        estimates[t] = prediction + kalman_gain * (series.iloc[t] - prediction)
        variances[t] = (1 - kalman_gain) * variance_prediction

    return pd.Series(estimates, index=series.index)

# Apply TMF and Kalman filter
def apply_indicators(data, length=21, process_noise=1e-5, measurement_noise=1e-2):
    def process_ticker(group):
        group['Kalman'] = calculate_kalman_filter_manual(
            group['close'], process_noise=process_noise, measurement_noise=measurement_noise
        )
        group = calculate_tmf(group, length=length)
        return group

    return data.groupby('Ticker').apply(process_ticker)

# Backtesting Strategy
def backtest_strategy(data):
    tradebook = []
    BROKERAGE = 0.001
    TRADE_CAPITAL = 100000
    TAKE_PROFIT = 0.03
    STOP_LOSS = 0.015

    cum_pnl = 0
    data['date'] = pd.to_datetime(data['date'])

    for ticker in data['Ticker'].unique():
        ticker_data = data[data['Ticker'] == ticker].copy().sort_values(by='date')

        position, entry_price, entry_time, quantity = None, 0, None, 0

        ticker_data['tp_price_long'] = ticker_data['close'] * (1 + TAKE_PROFIT)
        ticker_data['sl_price_long'] = ticker_data['close'] * (1 - STOP_LOSS)
        ticker_data['tp_price_short'] = ticker_data['close'] * (1 - TAKE_PROFIT)
        ticker_data['sl_price_short'] = ticker_data['close'] * (1 + STOP_LOSS)

        for i, row in ticker_data.iterrows():
            current_time = row['date'].time()

            if row['date'].weekday() in (0, 3):  # Skip Mondays and Thursdays
                continue

            if position is None and current_time <= pd.to_datetime("15:25").time():
                if row['TMF'] > 0.2499 and row['Kalman'] < row['close']:
                    position = 'long'
                    entry_price = row['close']
                    entry_time = row['date']
                    quantity = TRADE_CAPITAL / entry_price
                elif row['TMF'] < -0.2499 and row['Kalman'] > row['close']:
                    position = 'short'
                    entry_price = row['close']
                    entry_time = row['date']
                    quantity = TRADE_CAPITAL / entry_price

            if position is not None:
                exit_reason, exit_price = None, 0

                if position == 'long':
                    if row['high'] >= row['tp_price_long']:
                        exit_price = row['tp_price_long']
                        exit_reason = 'Take Profit'
                    elif row['low'] <= row['sl_price_long']:
                        exit_price = row['sl_price_long']
                        exit_reason = 'Stop Loss'
                elif position == 'short':
                    if row['low'] <= row['tp_price_short']:
                        exit_price = row['tp_price_short']
                        exit_reason = 'Take Profit'
                    elif row['high'] >= row['sl_price_short']:
                        exit_price = row['sl_price_short']
                        exit_reason = 'Stop Loss'

                if current_time.hour == 15 and current_time.minute == 28 and exit_reason is None:
                    exit_price = row['close']
                    exit_reason = 'End of Day'

                if exit_reason:
                    exit_time = row['date']
                    trade_cost = entry_price * quantity * BROKERAGE
                    exit_cost = exit_price * quantity * BROKERAGE
                    pnl = ((exit_price - entry_price) * quantity if position == 'long' else
                           (entry_price - exit_price) * quantity) - (trade_cost + exit_cost)

                    cum_pnl += pnl

                    tradebook.append({
                        'Ticker': ticker,
                        'Entry Time': entry_time,
                        'Exit Time': exit_time,
                        'Entry Price': entry_price,
                        'Exit Price': exit_price,
                        'PnL': pnl,
                        'Exit Reason': exit_reason,
                        'Cumulative PnL': cum_pnl
                    })

                    position, entry_price, entry_time, quantity = None, 0, None, 0

    tradebook_df = pd.DataFrame(tradebook)
    print(tradebook_df)
    tradebook_df.to_csv('/content/drive/MyDrive/tradebook.csv', index=False)

# Parameter Tuning with Progress Bar
def tune_parameters(data, process_noise_values, measurement_noise_values, length_values):
    results = []
    parameter_combinations = list(itertools.product(process_noise_values, measurement_noise_values, length_values))

    # Pre-sorting to improve performance during grouping
    data = data.sort_values(by=['Ticker', 'date'])

    with tqdm(total=len(parameter_combinations), desc="Tuning Parameters") as pbar:
        for process_noise, measurement_noise, length in parameter_combinations:
            data_with_indicators = apply_indicators(data.copy(), length=length,
                                                    process_noise=process_noise,
                                                    measurement_noise=measurement_noise)

            # Reset index to avoid ambiguity between 'Ticker' column and index
            data_with_indicators = data_with_indicators.reset_index(drop=True)

            tradebook = []
            BROKERAGE = 0.001
            TRADE_CAPITAL = 100000
            TAKE_PROFIT = 0.03
            STOP_LOSS = 0.015
            cum_pnl = 0

            # Process each ticker in the data
            for ticker, group in data_with_indicators.groupby('Ticker'):
                position = None
                entry_price = 0
                quantity = 0

                tp_price_long = group['close'] * (1 + TAKE_PROFIT)
                sl_price_long = group['close'] * (1 - STOP_LOSS)
                tp_price_short = group['close'] * (1 - TAKE_PROFIT)
                sl_price_short = group['close'] * (1 + STOP_LOSS)

                for _, row in group.iterrows():
                    if position is None:
                        if row['TMF'] > 0.2499 and row['Kalman'] < row['close']:
                            position = 'long'
                            entry_price = row['close']
                            quantity = TRADE_CAPITAL / entry_price
                        elif row['TMF'] < -0.2499 and row['Kalman'] > row['close']:
                            position = 'short'
                            entry_price = row['close']
                            quantity = TRADE_CAPITAL / entry_price
                    else:
                        exit_price, exit_reason = 0, None
                        if position == 'long':
                            if row['high'] >= tp_price_long.loc[row.name]:
                                exit_price = tp_price_long.loc[row.name]
                                exit_reason = 'Take Profit'
                            elif row['low'] <= sl_price_long.loc[row.name]:
                                exit_price = sl_price_long.loc[row.name]
                                exit_reason = 'Stop Loss'
                        elif position == 'short':
                            if row['low'] <= tp_price_short.loc[row.name]:
                                exit_price = tp_price_short.loc[row.name]
                                exit_reason = 'Take Profit'
                            elif row['high'] >= sl_price_short.loc[row.name]:
                                exit_price = sl_price_short.loc[row.name]
                                exit_reason = 'Stop Loss'

                        if exit_reason:
                            pnl = ((exit_price - entry_price) * quantity if position == 'long' else
                                   (entry_price - exit_price) * quantity)
                            pnl -= BROKERAGE * (entry_price + exit_price) * quantity
                            cum_pnl += pnl
                            tradebook.append({
                                'Ticker': ticker,
                                'PnL': pnl,
                                'Exit Reason': exit_reason
                            })
                            position = None

            # Performance Metrics
            tradebook_df = pd.DataFrame(tradebook)
            total_trades = len(tradebook_df)
            win_trades = len(tradebook_df[tradebook_df['PnL'] > 0])
            win_rate = (win_trades / total_trades) * 100 if total_trades > 0 else 0

            results.append({
                'Process Noise': process_noise,
                'Measurement Noise': measurement_noise,
                'Length': length,
                'Cumulative PnL': cum_pnl,
                'Win Rate': win_rate,
                'Total Trades': total_trades
            })

            pbar.update(1)

    # Convert to DataFrame for analysis
    results_df = pd.DataFrame(results)
    return results_df

# Example usage:
process_noise_values = [1e-5, 1e-6, 1e-4]
measurement_noise_values = [1e-2, 1e-3, 1e-1]
length_values = [14, 21, 28]

tuned_results = tune_parameters(data, process_noise_values, measurement_noise_values, length_values)
print(tuned_results)
